# ODLS-v2 on fastMRI knee multicoil (CORPD_FBK) -- Colab runner

Runs the **research variant** in `odls_v2/` (weight sharing across
phases, a denoiser prior replacing the low-rank module, and
cross-phase attention -- see `odls_v2/README.md` for the full
description), NOT the faithful baseline replication in `odls/`. This is
a separate notebook, with its own separate `CHECKPOINT_DIR`, so training
this never touches or overwrites your baseline's checkpoints.

Assumes you've already downloaded files from
https://www.kaggle.com/datasets/arafatshovon/fastmri-knee-multicoil
and uploaded them to Google Drive in **two separate folders**:
- 20 files for train/val (`DRIVE_DATA_DIR`)
- 4 files held out purely for testing (`DRIVE_TEST_DATA_DIR`)

(Same data as the baseline notebook -- if you already staged it for
that one in this same Colab session, this notebook reuses the same
local copies rather than re-downloading anything.)

Edit the **Config** cell below to match your Drive folder paths, then
run all cells top to bottom.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Config -- edit these paths

In [ ]:
# Folder in your Drive holding the 20 downloaded TRAIN/VAL files (loose
# .h5 files and/or a single archive containing them -- both handled below).
DRIVE_DATA_DIR = "/content/drive/MyDrive/odls_project/data/raw_corpd"

# Folder in your Drive holding the 4 dedicated TEST files (kept separate
# from the 20 above so the test score is on truly held-out data).
DRIVE_TEST_DATA_DIR = "/content/drive/MyDrive/odls_project/data/test_corpd"

# Where to stage local (fast-disk) copies on the Colab VM -- same paths
# as the baseline notebook on purpose, so if you already staged this
# data in this session, this notebook reuses it instead of re-copying.
LOCAL_RAW_DIR = "/content/fastmri_raw"
LOCAL_TRAIN_DIR = "/content/fastmri_corpd/train"
LOCAL_VAL_DIR = "/content/fastmri_corpd/val"
LOCAL_TEST_RAW_DIR = "/content/fastmri_test_raw"
LOCAL_TEST_DIR = "/content/fastmri_corpd/test"

REPO_URL = "https://github.com/Shambhawi419/1D_MRI.git"
REPO_DIR = "/content/1D_MRI"

# IMPORTANT: this must point INTO your Drive, not /content -- /content is
# local to the Colab VM and is wiped when the runtime disconnects, which
# would lose the trained weights. train.py writes a full-state checkpoint
# ("latest.pt") here every single epoch, so if training gets interrupted,
# simply re-running the Train cell below with the same CHECKPOINT_DIR
# resumes automatically from the last completed epoch instead of
# restarting at epoch 1.
#
# DELIBERATELY a different folder name ("odlsv2_checkpoints", not
# "odls_checkpoints_v3") from anything the baseline notebook uses -- this
# is a different model (ODLSv2, different architecture and parameter
# shapes), so its checkpoints could never be loaded by the baseline's
# code anyway, but keeping them in visibly separate folders avoids any
# confusion about which checkpoint belongs to which model.
CHECKPOINT_DIR = "/content/drive/MyDrive/odlsv2_checkpoints"

VAL_FRACTION = 0.2       # fraction of CORPD_FBK train/val files held out for validation
SPLIT_SEED = 0

N_COILS = 8               # must match N_VIRTUAL_COILS below
N_VIRTUAL_COILS = 8
MASK_TYPE = "cartesian"
AF = 4.0

# Forces every fastMRI slice -- from every file, train/val/test alike --
# to this exact (FE, PE) matrix size via a crop-or-pad round trip, both
# matching the paper's Sec. IV-A "224x224" preprocessing AND guaranteeing
# uniform shapes so DataLoader batching can never crash.
CROP_FE_TO = 224
CROP_PE_TO = 224

# Caps the shared soft-threshold's base value (starts at 0.001) -- same
# runaway-growth safeguard as the baseline; weight sharing doesn't remove
# the underlying incentive that caused it to explode there.
MAX_THRESHOLD = 0.05

# Dimension of the learnable per-phase embedding that FiLM-conditions the
# shared conv stacks and offsets the shared eta1/eta2/threshold scalars
# (weight sharing, direction 1 in odls_v2/README.md). Must match between
# training and evaluating the same checkpoint -- it's a model-construction
# hyperparameter, not something stored in the checkpoint's state_dict.
EMBED_DIM = 16

## Stage the data locally

Defines a reusable staging function and uses it to copy/extract both the
train/val set (`DRIVE_DATA_DIR` -> `LOCAL_RAW_DIR`) and the held-out test
set (`DRIVE_TEST_DATA_DIR` -> `LOCAL_TEST_RAW_DIR`), flattening any
`.zip`/`.tar*` archive found along the way so each local folder ends up
flat with `.h5` files regardless of how the files were stored in Drive.

In [ ]:
import os
import shutil
import tarfile
import zipfile


def stage_files(drive_dir, local_dir):
    """Copies loose .h5/.hdf5 files and extracts any .zip/.tar* archive
    from `drive_dir` into `local_dir`, flattening nested folders so
    `local_dir` ends up flat with .h5 files. Returns the sorted list of
    resulting .h5 filenames."""
    os.makedirs(local_dir, exist_ok=True)

    if not os.path.isdir(drive_dir):
        raise FileNotFoundError(
            f"{drive_dir} not found -- check the Drive is mounted and the "
            "folder path/name is correct."
        )

    n_copied, n_extracted = 0, 0
    for name in sorted(os.listdir(drive_dir)):
        src = os.path.join(drive_dir, name)
        if os.path.isdir(src):
            continue

        if name.lower().endswith((".h5", ".hdf5")):
            dst = os.path.join(local_dir, name)
            if not os.path.exists(dst):
                shutil.copy2(src, dst)
            n_copied += 1

        elif zipfile.is_zipfile(src):
            with zipfile.ZipFile(src) as zf:
                zf.extractall(local_dir)
            n_extracted += 1

        elif tarfile.is_tarfile(src):
            with tarfile.open(src) as tf:
                tf.extractall(local_dir)
            n_extracted += 1

    # Archives sometimes unpack into a nested subfolder -- flatten any
    # .h5 files found below local_dir up to its top level.
    for root, _, files in os.walk(local_dir):
        if root == local_dir:
            continue
        for f in files:
            if f.lower().endswith((".h5", ".hdf5")):
                src = os.path.join(root, f)
                dst = os.path.join(local_dir, f)
                if not os.path.exists(dst):
                    shutil.move(src, dst)

    h5_files = sorted(
        f for f in os.listdir(local_dir) if f.lower().endswith((".h5", ".hdf5"))
    )
    print(f"{drive_dir}: copied {n_copied} loose .h5 files, extracted {n_extracted} archive(s)")
    print(f"{len(h5_files)} .h5 files staged locally in {local_dir}")
    return h5_files


h5_files = stage_files(DRIVE_DATA_DIR, LOCAL_RAW_DIR)
test_h5_files = stage_files(DRIVE_TEST_DATA_DIR, LOCAL_TEST_RAW_DIR)

## Clone the repo and filter to CORPD_FBK

In [ ]:
import subprocess

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

import sys
# odls_v2, NOT odls -- this notebook always imports the research variant's
# code, so anything run interactively here (e.g. a diagnostic cell) uses
# the same ODLSv2 model the Train/Test cells below use.
sys.path.insert(0, os.path.join(REPO_DIR, "odls_v2"))

from fastmri_data import find_corpd_files

corpd_files = find_corpd_files(LOCAL_RAW_DIR, fat_suppressed=False)
print(f"train/val pool: {len(corpd_files)} of {len(h5_files)} staged files are CORPD_FBK")
for p in corpd_files:
    print(" ", os.path.basename(p))

if not corpd_files:
    raise RuntimeError(
        "No CORPD_FBK files found among the staged train/val data -- if "
        "your files are the fat-suppressed variant instead, re-run with "
        "fat_suppressed=True, or use --fastmri-fat-suppressed in the "
        "training cell below."
    )

test_corpd_files = find_corpd_files(LOCAL_TEST_RAW_DIR, fat_suppressed=False)
print(f"\ntest pool: {len(test_corpd_files)} of {len(test_h5_files)} staged files are CORPD_FBK")
for p in test_corpd_files:
    print(" ", os.path.basename(p))

if not test_corpd_files:
    raise RuntimeError(
        "No CORPD_FBK files found among the staged test data -- check "
        "DRIVE_TEST_DATA_DIR actually contains the 4 test files."
    )

## Split into train/val/test folders

`train.py` / `evaluate.py` take a directory each, so the CORPD_FBK files
found above are split and symlinked into three separate folders: the
train/val pool is split by `VAL_FRACTION`, and the dedicated test pool
(your 4 held-out files) goes into its own folder untouched.

In [ ]:
import random

os.makedirs(LOCAL_TRAIN_DIR, exist_ok=True)
os.makedirs(LOCAL_VAL_DIR, exist_ok=True)
os.makedirs(LOCAL_TEST_DIR, exist_ok=True)

rng = random.Random(SPLIT_SEED)
shuffled = corpd_files[:]
rng.shuffle(shuffled)

n_val = max(1, int(round(len(shuffled) * VAL_FRACTION))) if len(shuffled) > 1 else 0
val_files = shuffled[:n_val]
train_files = shuffled[n_val:]

def _relink(files, dest_dir):
    for src in files:
        link_path = os.path.join(dest_dir, os.path.basename(src))
        if os.path.lexists(link_path):
            os.remove(link_path)
        os.symlink(src, link_path)

_relink(train_files, LOCAL_TRAIN_DIR)
_relink(val_files, LOCAL_VAL_DIR)
_relink(test_corpd_files, LOCAL_TEST_DIR)

print(f"train: {len(train_files)} files -> {LOCAL_TRAIN_DIR}")
print(f"val:   {len(val_files)} files -> {LOCAL_VAL_DIR}")
print(f"test:  {len(test_corpd_files)} files -> {LOCAL_TEST_DIR}")

## Train (ODLS-v2)

Runs `odls_v2/train.py`, NOT the baseline's. Progress bars (via tqdm)
show per-epoch and per-batch loss as it runs. Checkpoints go straight to
`CHECKPOINT_DIR` on your Drive every epoch, so if this cell's runtime
disconnects partway through, just re-run this same cell -- it
automatically resumes from the last completed epoch instead of starting
over.

`--num-workers 2` matches Colab's typical 2-CPU allocation -- raising it
past your actual CPU count causes worker contention that starves the GPU
rather than feeding it faster.

In [ ]:
!cd {REPO_DIR}/odls_v2 && python train.py \
    --fastmri-train-root {LOCAL_TRAIN_DIR} \
    --fastmri-val-root {LOCAL_VAL_DIR} \
    --n-coils {N_COILS} --n-virtual-coils {N_VIRTUAL_COILS} \
    --crop-fe-to {CROP_FE_TO} --crop-pe-to {CROP_PE_TO} \
    --max-threshold {MAX_THRESHOLD} --embed-dim {EMBED_DIM} \
    --mask-type {MASK_TYPE} --af {AF} \
    --num-workers 2 \
    --checkpoint-dir {CHECKPOINT_DIR}

## Test (ODLS-v2)

Scores the best checkpoint (`odls_best.pt`, from `CHECKPOINT_DIR` on your
Drive) against the 4 dedicated test files staged in `LOCAL_TEST_DIR`
earlier -- truly held-out data, never seen during training or
validation.

In [ ]:
!cd {REPO_DIR}/odls_v2 && python evaluate.py \
    --fastmri-test-root {LOCAL_TEST_DIR} \
    --checkpoint {CHECKPOINT_DIR}/odls_best.pt \
    --n-coils {N_COILS} --n-virtual-coils {N_VIRTUAL_COILS} \
    --crop-fe-to {CROP_FE_TO} --crop-pe-to {CROP_PE_TO} \
    --max-threshold {MAX_THRESHOLD} --embed-dim {EMBED_DIM} \
    --mask-type {MASK_TYPE} --af {AF}